Shared setup (run once):

In [1]:
import os, json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

FEATURES_PATH = "../data/processed/features.csv"
MODEL_DIR = "../model"
TEST_SIZE = 0.25
SEED = 42
META_COLS = ["source_file", "label", "subject_id", "session_id", "payload_variant", "window_start_seq"]
SENTINEL = -999
os.makedirs(MODEL_DIR, exist_ok=True)

df = pd.read_csv(FEATURES_PATH)
feature_cols = [c for c in df.columns if c not in META_COLS]

X = df[feature_cols].fillna(SENTINEL)
y = (df["label"] == "injected").astype(int)

# Subject-independent for human data, session-independent for injected data
# (only one physical injected-source subject exists) -- same logic as notebook 2.
groups = pd.Series(
    np.where(df["label"] == "human", df["subject_id"], df["session_id"]),
    index=df.index,
)

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
test_meta = df.iloc[test_idx]

def evaluate_and_save(clf, name, filename):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else y_pred.astype(float)

    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=["human", "injected"], zero_division=0))
    print(confusion_matrix(y_test, y_pred))
    print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.4f}")

    tmp = test_meta.copy()
    tmp["y_true"], tmp["y_pred"] = y_test.values, y_pred
    sess = tmp.groupby("session_id").agg(y_true=("y_true", "first"), pred_rate=("y_pred", "mean"))
    sess["y_pred"] = (sess["pred_rate"] >= 0.5).astype(int)
    sess_acc = (sess["y_true"] == sess["y_pred"]).mean()
    print(f"Session-level accuracy: {sess_acc:.4f}\n")

    path = os.path.join(MODEL_DIR, filename)
    joblib.dump(clf, path)
    print(f"Saved -> {path}\n")
    return clf

Naive Bayes:

In [2]:
from sklearn.naive_bayes import GaussianNB
clf_nb = evaluate_and_save(GaussianNB(), "Naive Bayes", "naive_bayes_model.joblib")

=== Naive Bayes ===
              precision    recall  f1-score   support

       human       0.97      1.00      0.98      1437
    injected       1.00      0.98      0.99      2092

    accuracy                           0.99      3529
   macro avg       0.98      0.99      0.99      3529
weighted avg       0.99      0.99      0.99      3529

[[1437    0]
 [  45 2047]]
ROC AUC: 0.9814
Session-level accuracy: 0.9730

Saved -> ../model\naive_bayes_model.joblib



K-Nearest Neighbors:

In [3]:
from sklearn.neighbors import KNeighborsClassifier
clf_knn = evaluate_and_save(
    make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
    "K-Nearest Neighbors", "knn_model.joblib"
)

=== K-Nearest Neighbors ===
              precision    recall  f1-score   support

       human       1.00      0.97      0.98      1437
    injected       0.98      1.00      0.99      2092

    accuracy                           0.99      3529
   macro avg       0.99      0.98      0.98      3529
weighted avg       0.99      0.99      0.99      3529

[[1387   50]
 [   1 2091]]
ROC AUC: 0.9982
Session-level accuracy: 1.0000

Saved -> ../model\knn_model.joblib



Decision Tree:

In [4]:
from sklearn.tree import DecisionTreeClassifier
clf_dt = evaluate_and_save(
    DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=SEED),
    "Decision Tree", "decision_tree_model.joblib"
)

=== Decision Tree ===
              precision    recall  f1-score   support

       human       1.00      0.92      0.96      1437
    injected       0.95      1.00      0.97      2092

    accuracy                           0.97      3529
   macro avg       0.97      0.96      0.96      3529
weighted avg       0.97      0.97      0.97      3529

[[1318  119]
 [   2 2090]]
ROC AUC: 0.9953
Session-level accuracy: 1.0000

Saved -> ../model\decision_tree_model.joblib



Logistic Regression:

In [5]:
from sklearn.linear_model import LogisticRegression
clf_lr = evaluate_and_save(
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")),
    "Logistic Regression", "logistic_regression_model.joblib"
)

=== Logistic Regression ===
              precision    recall  f1-score   support

       human       0.96      0.98      0.97      1437
    injected       0.99      0.98      0.98      2092

    accuracy                           0.98      3529
   macro avg       0.98      0.98      0.98      3529
weighted avg       0.98      0.98      0.98      3529

[[1412   25]
 [  52 2040]]
ROC AUC: 0.9902
Session-level accuracy: 0.9730

Saved -> ../model\logistic_regression_model.joblib



HistGradientBoosting:

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier
clf_hgb = evaluate_and_save(
    HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED),
    "HistGradientBoosting", "histgb_model.joblib"
)

=== HistGradientBoosting ===
              precision    recall  f1-score   support

       human       1.00      0.96      0.98      1437
    injected       0.98      1.00      0.99      2092

    accuracy                           0.99      3529
   macro avg       0.99      0.98      0.98      3529
weighted avg       0.99      0.99      0.99      3529

[[1386   51]
 [   0 2092]]
ROC AUC: 0.9996
Session-level accuracy: 1.0000

Saved -> ../model\histgb_model.joblib



XGBoost:

In [7]:
from xgboost import XGBClassifier
# XGBoost has no class_weight param -- scale_pos_weight is the equivalent
# (ratio of negative/human to positive/injected examples in the training split).
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
clf_xgb = evaluate_and_save(
    XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1, eval_metric="logloss",
                  scale_pos_weight=scale_pos_weight, random_state=SEED, n_jobs=-1),
    "XGBoost", "xgboost_model.joblib"
)

=== XGBoost ===
              precision    recall  f1-score   support

       human       1.00      0.98      0.99       897
    injected       0.99      1.00      1.00      2079

    accuracy                           0.99      2976
   macro avg       0.99      0.99      0.99      2976
weighted avg       0.99      0.99      0.99      2976

[[ 882   15]
 [   4 2075]]
ROC AUC: 0.9998
Session-level accuracy: 1.0000

Saved -> model\xgboost_model.joblib

